In [ ]:
from pathlib import Path
import re

import pymupdf
import pymupdf4llm


PDF_PATH = Path("../../../datasets/WattBot2026/data/fritze2021.pdf")

OUTPUT_DIR = Path("output/fritze2021")
ASSETS_DIR = OUTPUT_DIR / "assets"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
ASSETS_DIR.mkdir(parents=True, exist_ok=True)


def render_vector_figure(doc, page_number, bbox, output_path):
    page = doc[page_number]

    clip = pymupdf.Rect(bbox)
    clip = pymupdf.Rect(
        max(page.rect.x0, clip.x0 - 2),
        max(page.rect.y0, clip.y0 - 2),
        min(page.rect.x1, clip.x1 + 2),
        min(page.rect.y1, clip.y1 + 2),
    )

    temp_doc = pymupdf.open()

    temp_page = temp_doc.new_page(
        width=clip.width,
        height=clip.height,
    )

    temp_page.show_pdf_page(
        temp_page.rect,
        doc,
        page_number,
        clip=clip,
    )

    svg = temp_page.get_svg_image(
        text_as_path=True
    )

    output_path.write_text(
        svg,
        encoding="utf-8",
    )

    temp_doc.close()


# ------------------------------------------------------------
# Extract Markdown + temporary images + layout information
# ------------------------------------------------------------

chunks = pymupdf4llm.to_markdown(
    str(PDF_PATH),
    page_chunks=True,
    write_images=False,
    show_progress=True,
)


doc = pymupdf.open(str(PDF_PATH))

markdown_pages = []

figure_counter = 0
temporary_pngs = []


# ------------------------------------------------------------
# Replace rasterized vector figures with SVG figures
# ------------------------------------------------------------

for chunk_index, chunk in enumerate(chunks):

    page_number = chunk.get("metadata", {}).get(
        "page_number",
        chunk_index + 1,
    )

    page_index = page_number - 1

    text = chunk["text"]

    picture_boxes = [
        box
        for box in chunk.get("page_boxes", [])
        if box.get("class") == "picture"
    ]

    # Process from bottom to top so character positions remain valid.
    picture_boxes.sort(
        key=lambda box: box.get("pos", (0, 0))[0],
        reverse=True,
    )

    for box in picture_boxes:

        start, end = box.get("pos", (0, 0))

        if end <= start:
            continue

        original_block = text[start:end]

        image_matches = re.findall(
            r"!\[[^\]]*\]\(([^)]+)\)",
            original_block,
        )

        bbox = box.get("bbox")

        if not bbox or len(bbox) != 4:
            continue

        figure_counter += 1

        svg_name = (
            f"figure_{figure_counter:04d}"
            f"_page_{page_number:04d}.svg"
        )

        svg_path = ASSETS_DIR / svg_name

        render_vector_figure(
            doc,
            page_index,
            bbox,
            svg_path,
        )

        text = (
            text[:start]
            + f"![Figure](assets/{svg_name})"
            + text[end:]
        )

        for image_path in image_matches:

            filename = Path(
                image_path.replace("\\", "/")
            ).name

            png_path = ASSETS_DIR / filename

            if png_path.exists():
                temporary_pngs.append(png_path)

    markdown_pages.append(text)


doc.close()


# ------------------------------------------------------------
# Combine pages
# ------------------------------------------------------------

markdown = "\n\n".join(markdown_pages)


# ------------------------------------------------------------
# Normalize remaining image references
# ------------------------------------------------------------

def normalize_image_path(match):
    alt = match.group(1)
    image_path = match.group(2)

    filename = Path(
        image_path.replace("\\", "/")
    ).name

    local_file = ASSETS_DIR / filename

    if local_file.exists():
        return f"![{alt}](assets/{filename})"

    return match.group(0)


markdown = re.sub(
    r"!\[([^\]]*)\]\(([^)]+)\)",
    normalize_image_path,
    markdown,
)


# ------------------------------------------------------------
# Remove the low-quality temporary raster versions of
# vector figures after replacing them with SVG.
# ------------------------------------------------------------

for png_path in temporary_pngs:
    if png_path.exists():
        png_path.unlink()


# ------------------------------------------------------------
# Save Markdown
# ------------------------------------------------------------

OUTPUT_FILE = OUTPUT_DIR / "document.md"

OUTPUT_FILE.write_text(
    markdown,
    encoding="utf-8",
)


print(f"Done: {OUTPUT_FILE}")
print(f"Assets: {ASSETS_DIR}")



AttributeError: module 'pymupdf4llm' has no attribute 'helpers'

In [3]:
from pathlib import Path
import re
import shutil
import xml.etree.ElementTree as ET

import pymupdf
import pymupdf.layout
import pymupdf4llm

# ============================================================
# Configuration
# ============================================================

DATA_DIR = Path(r"D:\machine learning\datasets\WattBot2026\temp")

OUTPUT_ROOT = DATA_DIR.parent / "processed_markdown"


# ============================================================
# Mathematical symbols
#
# IMPORTANT:
# Keep Greek letters as Unicode.
# KaTeX supports direct Unicode Greek input.
#
# Symbols that benefit from LaTeX are converted to commands
# terminated with {} so they can NEVER become things like:
#
#     \thetac
#     \PhiR
# ============================================================

SYMBOL_TO_LATEX = {
    "×": r"\times{}",
    "÷": r"\div{}",
    "±": r"\pm{}",
    "∓": r"\mp{}",
    "≤": r"\leq{}",
    "≥": r"\geq{}",
    "≠": r"\neq{}",
    "≈": r"\approx{}",
    "≃": r"\simeq{}",
    "≡": r"\equiv{}",
    "∝": r"\propto{}",
    "∞": r"\infty{}",
    "∑": r"\sum{}",
    "∏": r"\prod{}",
    "∫": r"\int{}",
    "∬": r"\iint{}",
    "∭": r"\iiint{}",
    "∮": r"\oint{}",
    "∂": r"\partial{}",
    "∇": r"\nabla{}",
    "√": r"\sqrt{}",
    "∈": r"\in{}",
    "∉": r"\notin{}",
    "⊂": r"\subset{}",
    "⊆": r"\subseteq{}",
    "⊃": r"\supset{}",
    "⊇": r"\supseteq{}",
    "∪": r"\cup{}",
    "∩": r"\cap{}",
    "∅": r"\emptyset{}",
    "→": r"\rightarrow{}",
    "←": r"\leftarrow{}",
    "↔": r"\leftrightarrow{}",
    "⇒": r"\Rightarrow{}",
    "⇐": r"\Leftarrow{}",
    "⇔": r"\Leftrightarrow{}",
    "·": r"\cdot{}",
    # PDF minus / dash variants
    "−": "-",
    "–": "-",
    "—": "-",
}


# ============================================================
# Unicode super/subscripts
# ============================================================

SUPERSCRIPT = {
    "⁰": "0",
    "¹": "1",
    "²": "2",
    "³": "3",
    "⁴": "4",
    "⁵": "5",
    "⁶": "6",
    "⁷": "7",
    "⁸": "8",
    "⁹": "9",
    "⁺": "+",
    "⁻": "-",
    "⁼": "=",
    "⁽": "(",
    "⁾": ")",
    "ⁿ": "n",
    "ⁱ": "i",
}

SUBSCRIPT = {
    "₀": "0",
    "₁": "1",
    "₂": "2",
    "₃": "3",
    "₄": "4",
    "₅": "5",
    "₆": "6",
    "₇": "7",
    "₈": "8",
    "₉": "9",
    "₊": "+",
    "₋": "-",
    "₌": "=",
    "₍": "(",
    "₎": ")",
    "ₐ": "a",
    "ₑ": "e",
    "ₕ": "h",
    "ᵢ": "i",
    "ⱼ": "j",
    "ₖ": "k",
    "ₗ": "l",
    "ₘ": "m",
    "ₙ": "n",
    "ₒ": "o",
    "ₚ": "p",
    "ᵣ": "r",
    "ₛ": "s",
    "ₜ": "t",
    "ᵤ": "u",
    "ᵥ": "v",
    "ₓ": "x",
}


def convert_unicode_scripts(text):
    """Convert Unicode superscript/subscript runs to LaTeX."""

    result = []
    i = 0

    while i < len(text):

        char = text[i]

        if char in SUPERSCRIPT:

            value = []

            while i < len(text) and text[i] in SUPERSCRIPT:
                value.append(SUPERSCRIPT[text[i]])
                i += 1

            result.append("^{" + "".join(value) + "}")

            continue

        if char in SUBSCRIPT:

            value = []

            while i < len(text) and text[i] in SUBSCRIPT:
                value.append(SUBSCRIPT[text[i]])
                i += 1

            result.append("_{" + "".join(value) + "}")

            continue

        result.append(char)
        i += 1

    return "".join(result)


# ============================================================
# Formula formatting
# ============================================================


def format_formula(text):
    """
    Turn PDF-extracted formula text into KaTeX-safe
    Markdown display math.

    We deliberately keep Greek letters as Unicode.
    """

    if not text:
        return ""

    # Normalize Unicode.
    text = text.replace("\uf0b7", "·")
    text = text.replace("\u00a0", " ")

    # Normalize whitespace.
    text = re.sub(
        r"\s+",
        " ",
        text,
    ).strip()

    # --------------------------------------------------------
    # Dot above a variable.
    #
    # Example:
    #     ˙θ
    #
    # becomes:
    #     \dot{θ}
    # --------------------------------------------------------

    text = re.sub(
        r"˙\s*([A-Za-zΑ-Ωα-ω])",
        r"\\dot{\1}",
        text,
    )

    # --------------------------------------------------------
    # Convert mathematical Unicode operators.
    # --------------------------------------------------------

    for source, target in SYMBOL_TO_LATEX.items():
        text = text.replace(
            source,
            target,
        )

    # --------------------------------------------------------
    # Unicode superscripts/subscripts.
    # --------------------------------------------------------

    text = convert_unicode_scripts(text)

    # --------------------------------------------------------
    # Characters that have special meaning in LaTeX.
    # --------------------------------------------------------

    text = text.replace(
        "\\",
        r"\backslash{}",
    )

    # Restore our generated commands.
    text = text.replace(
        r"\backslash{}dot",
        r"\dot",
    )

    for command in [
        r"\times",
        r"\div",
        r"\pm",
        r"\mp",
        r"\leq",
        r"\geq",
        r"\neq",
        r"\approx",
        r"\simeq",
        r"\equiv",
        r"\propto",
        r"\infty",
        r"\sum",
        r"\prod",
        r"\int",
        r"\iint",
        r"\iiint",
        r"\oint",
        r"\partial",
        r"\nabla",
        r"\sqrt",
        r"\in",
        r"\notin",
        r"\subset",
        r"\subseteq",
        r"\supset",
        r"\supseteq",
        r"\cup",
        r"\cap",
        r"\emptyset",
        r"\rightarrow",
        r"\leftarrow",
        r"\leftrightarrow",
        r"\Rightarrow",
        r"\Leftarrow",
        r"\Leftrightarrow",
        r"\cdot",
    ]:
        text = text.replace(
            r"\backslash{}" + command[1:],
            command + "{}",
        )

    # Restore \dot.
    text = text.replace(
        r"\backslash{}dot",
        r"\dot",
    )

    # Escape Markdown / LaTeX special characters
    # only when they occur literally.
    text = text.replace(
        "&",
        r"\&",
    )

    text = text.replace(
        "#",
        r"\#",
    )

    # Never allow a raw '%' to start a LaTeX comment.
    text = text.replace(
        "%",
        r"\%",
    )

    text = re.sub(
        r"\s+",
        " ",
        text,
    ).strip()

    if not text:
        return ""

    return "\n\n$$\n" + text + "\n$$\n\n"


# ============================================================
# Decide whether a "formula" box really looks mathematical
#
# PyMuPDF Layout can classify ordinary prose as "formula".
# This is a known behavior, so don't blindly wrap all
# formula boxes in KaTeX.
# ============================================================


def looks_like_formula(text):

    if not text:
        return False

    text = text.strip()

    # Strong mathematical indicators.
    if re.search(
        r"[=<>±×÷∑∏∫√∂∇∞≤≥≠≈∝∈∉∪∩]",
        text,
    ):
        return True

    if "^" in text or "_" in text:
        return True

    if re.search(
        r"[A-Za-z]\s*/\s*[A-Za-z0-9]",
        text,
    ):
        return True

    # Greek letters strongly suggest mathematics.
    if re.search(
        r"[α-ωΑ-Ω]",
        text,
    ):
        return True

    # Short compact expressions such as:
    #     r = d / 2
    #     y = mx + c
    tokens = text.split()

    if len(tokens) <= 8 and re.search(
        r"\b[A-Za-z]\s*=",
        text,
    ):
        return True

    # Otherwise likely ordinary prose.
    return False


# ============================================================
# Extract formula text from the actual PDF
# ============================================================


def extract_formula_text(page, bbox):

    rect = pymupdf.Rect(bbox)

    text = page.get_textbox(rect).strip()

    if text:
        return text

    # Fallback.
    return page.get_text(
        "text",
        clip=rect,
        sort=True,
    ).strip()


# ============================================================
# SVG duplicate-text cleanup
# ============================================================


def remove_duplicate_svg_text(svg):

    try:
        root = ET.fromstring(svg)
    except ET.ParseError:
        return svg

    SVG_NS = "http://www.w3.org/2000/svg"

    XLINK_NS = "http://www.w3.org/1999/xlink"

    ET.register_namespace(
        "",
        SVG_NS,
    )

    ET.register_namespace(
        "xlink",
        XLINK_NS,
    )

    seen = set()

    def inside_defs(element):
        current = element

        while current is not None:
            tag = current.tag

            if isinstance(tag, str):
                local = tag.rsplit(
                    "}",
                    1,
                )[-1]

                if local == "defs":
                    return True

            current = parent_map.get(current)

        return False

    # Build parent lookup.
    parent_map = {child: parent for parent in root.iter() for child in parent}

    for parent in list(root.iter()):

        for child in list(parent):

            tag = child.tag

            if not isinstance(tag, str):
                continue

            tag = tag.rsplit(
                "}",
                1,
            )[-1]

            # We only touch rendered text objects.
            if tag not in {
                "text",
                "use",
            }:
                continue

            # Do not touch definitions.
            if inside_defs(child):
                continue

            attributes = tuple(
                sorted(
                    (
                        str(k),
                        str(v),
                    )
                    for k, v in child.attrib.items()
                )
            )

            visible_text = (
                child.attrib.get(
                    "data-text",
                    "",
                )
                or "".join(child.itertext()).strip()
            )

            # For <use>, x/y/transform/href identify
            # the exact rendered object.
            key = (
                tag,
                visible_text,
                attributes,
            )

            if key in seen:
                parent.remove(child)
            else:
                seen.add(key)

    return ET.tostring(
        root,
        encoding="unicode",
    )


# ============================================================
# Render one picture region to SVG
# ============================================================


def render_picture_as_svg(
    doc,
    page_number,
    bbox,
    output_path,
):

    page = doc[page_number]

    clip = pymupdf.Rect(bbox)

    # Small padding.
    clip = pymupdf.Rect(
        max(
            page.cropbox.x0,
            clip.x0 - 2,
        ),
        max(
            page.cropbox.y0,
            clip.y0 - 2,
        ),
        min(
            page.cropbox.x1,
            clip.x1 + 2,
        ),
        min(
            page.cropbox.y1,
            clip.y1 + 2,
        ),
    )

    temp_doc = pymupdf.open()

    try:

        temp_page = temp_doc.new_page(
            width=clip.width,
            height=clip.height,
        )

        # Use the original page only for this clipped region.
        temp_page.show_pdf_page(
            temp_page.rect,
            doc,
            page_number,
            clip=clip,
        )

        # IMPORTANT:
        #
        # text_as_path=False gives normal SVG text objects.
        # This allows us to identify and remove exact duplicate
        # text objects.
        svg = temp_page.get_svg_image(
            text_as_path=False,
        )

        svg = remove_duplicate_svg_text(svg)

        output_path.write_text(
            svg,
            encoding="utf-8",
        )

    finally:
        temp_doc.close()


# ============================================================
# Determine where a box belongs in Markdown
# ============================================================


def get_box_position(
    box,
    boxes,
    text_length,
):

    start, end = box.get(
        "pos",
        (0, 0),
    )

    # Normal box with a valid text range.
    if end > start:
        return start

    index = box.get(
        "index",
        0,
    )

    previous_end = None
    next_start = None

    for other in boxes:

        other_index = other.get(
            "index",
            0,
        )

        if other_index >= index:
            continue

        other_start, other_end = other.get(
            "pos",
            (0, 0),
        )

        if other_end > other_start:

            if previous_end is None or other_end > previous_end:
                previous_end = other_end

    for other in boxes:

        other_index = other.get(
            "index",
            0,
        )

        if other_index <= index:
            continue

        other_start, other_end = other.get(
            "pos",
            (0, 0),
        )

        if other_end > other_start:

            if next_start is None or other_start < next_start:
                next_start = other_start

    if previous_end is not None:
        return previous_end

    if next_start is not None:
        return next_start

    return text_length


# ============================================================
# Process one PDF
# ============================================================


def process_pdf(pdf_path):

    print(f"\nProcessing: {pdf_path.name}")

    output_dir = OUTPUT_ROOT / pdf_path.stem

    assets_dir = output_dir / "assets"

    # Remove old generated output.
    if output_dir.exists():
        shutil.rmtree(output_dir)

    assets_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    doc = pymupdf.open(str(pdf_path))

    figure_counter = 0
    formula_counter = 0

    try:

        # ----------------------------------------------------
        # Layout extraction.
        #
        # write_images=False is intentional.
        # We create our own SVGs.
        #
        # force_text=False avoids asking 4LLM to force text
        # from picture regions.
        # ----------------------------------------------------

        chunks = pymupdf4llm.to_markdown(
            doc,
            page_chunks=True,
            write_images=False,
            force_text=False,
            show_progress=True,
        )

        markdown_pages = []

        for chunk_index, chunk in enumerate(chunks):

            metadata = chunk.get(
                "metadata",
                {},
            )

            page_number = metadata.get(
                "page_number",
                chunk_index + 1,
            )

            page_index = page_number - 1

            page = doc[page_index]

            text = chunk.get(
                "text",
                "",
            )

            boxes = sorted(
                chunk.get(
                    "page_boxes",
                    [],
                ),
                key=lambda box: box.get(
                    "index",
                    0,
                ),
            )

            replacements = []

            # =================================================
            # Process layout boxes
            # =================================================

            for box in boxes:

                box_class = box.get("class")

                bbox = box.get("bbox")

                if not bbox or len(bbox) != 4:
                    continue

                # =============================================
                # PICTURE
                # =============================================

                if box_class == "picture":

                    figure_counter += 1

                    svg_name = (
                        f"figure_"
                        f"{figure_counter:04d}"
                        f"_page_"
                        f"{page_number:04d}"
                        f".svg"
                    )

                    svg_path = assets_dir / svg_name

                    render_picture_as_svg(
                        doc,
                        page_index,
                        bbox,
                        svg_path,
                    )

                    start, end = box.get(
                        "pos",
                        (0, 0),
                    )

                    # The old code kept original_block.
                    #
                    # That was exactly what caused:
                    #
                    #   image text
                    #   +
                    #   same text in Markdown
                    #
                    # Now the picture block itself is replaced
                    # by ONLY the SVG reference.

                    if end > start:

                        replacement = f"![Figure]" f"(assets/{svg_name})"

                        replacements.append(
                            (
                                start,
                                end,
                                replacement,
                                box.get(
                                    "index",
                                    0,
                                ),
                            )
                        )

                    else:

                        position = get_box_position(
                            box,
                            boxes,
                            len(text),
                        )

                        replacement = "\n\n" f"![Figure]" f"(assets/{svg_name})" "\n\n"

                        replacements.append(
                            (
                                position,
                                position,
                                replacement,
                                box.get(
                                    "index",
                                    0,
                                ),
                            )
                        )

                # =============================================
                # FORMULA
                # =============================================

                elif box_class == "formula":

                    formula_text = extract_formula_text(
                        page,
                        bbox,
                    )

                    if not formula_text:
                        continue

                    formula_counter += 1

                    # -----------------------------------------
                    # Important:
                    #
                    # The layout classifier sometimes labels
                    # ordinary prose as "formula".
                    #
                    # Therefore only actual-looking formulas
                    # are wrapped in $$.
                    # -----------------------------------------

                    if looks_like_formula(formula_text):

                        replacement = format_formula(formula_text)

                    else:

                        # Misclassified ordinary prose.
                        replacement = "\n\n" + formula_text.strip() + "\n\n"

                    start, end = box.get(
                        "pos",
                        (0, 0),
                    )

                    if end > start:

                        replacements.append(
                            (
                                start,
                                end,
                                replacement,
                                box.get(
                                    "index",
                                    0,
                                ),
                            )
                        )

                    else:

                        position = get_box_position(
                            box,
                            boxes,
                            len(text),
                        )

                        replacements.append(
                            (
                                position,
                                position,
                                replacement,
                                box.get(
                                    "index",
                                    0,
                                ),
                            )
                        )

            # =================================================
            # Apply replacements right -> left
            # =================================================

            replacements.sort(
                key=lambda item: (
                    item[0],
                    item[1],
                    item[3],
                )
            )

            for (
                start,
                end,
                replacement,
                _,
            ) in reversed(replacements):

                start = max(
                    0,
                    min(
                        start,
                        len(text),
                    ),
                )

                end = max(
                    start,
                    min(
                        end,
                        len(text),
                    ),
                )

                text = text[:start] + replacement + text[end:]

            markdown_pages.append(text)

        # ====================================================
        # Combine pages
        # ====================================================

        markdown = "\n\n".join(markdown_pages)

        output_file = output_dir / "document.md"

        output_file.write_text(
            markdown,
            encoding="utf-8",
        )

    finally:
        doc.close()

    print(f"Done: {pdf_path.name}")

    print(f"  Figures  : {figure_counter}")

    print(f"  Formulas : {formula_counter}")

    print(f"  Markdown : {output_file}")

    return output_file


# ============================================================
# Process every PDF
# ============================================================


def main():

    OUTPUT_ROOT.mkdir(
        parents=True,
        exist_ok=True,
    )

    pdf_files = sorted(
        path
        for path in DATA_DIR.iterdir()
        if path.is_file() and path.suffix.lower() == ".pdf"
    )

    if not pdf_files:

        print(f"No PDF files found in:\n{DATA_DIR}")

        return

    print(f"Found {len(pdf_files)} PDF(s).")

    successful = 0
    failed = []

    for index, pdf_path in enumerate(
        pdf_files,
        start=1,
    ):

        print("\n" + "=" * 70)

        print(f"[{index}/{len(pdf_files)}] " f"{pdf_path.name}")

        print("=" * 70)

        try:

            process_pdf(pdf_path)

            successful += 1

        except Exception as exc:

            failed.append(
                (
                    pdf_path.name,
                    str(exc),
                )
            )

            print(f"FAILED: {pdf_path.name}")

            print(f"ERROR: {exc}")

    print("\n" + "=" * 70)

    print("BATCH COMPLETE")

    print("=" * 70)

    print(f"Total     : {len(pdf_files)}")

    print(f"Successful: {successful}")

    print(f"Failed    : {len(failed)}")

    if failed:

        print("\nFailed files:")

        for filename, error in failed:

            print(f"- {filename}: {error}")


# ============================================================
# Run
# ============================================================

if __name__ == "__main__":
    main()

Found 4 PDF(s).

[1/4] dutt2025.pdf

Processing: dutt2025.pdf
Parsing 19 pages of 'D:\machine learning\datasets\WattBot2026\temp\dutt2025.pdf'...


100%|██████████| 19/19 [00:24<00:00,  1.31s/it]


=== Document parser messages ===
Using RapidOCR for OCR processing.
OCR on page.number=4/5.
OCR on page.number=13/14.
OCR on page.number=16/17.
OCR on page.number=17/18.
 on page.number=14/15.


Generating markdown text...


100%|██████████| 19/19 [00:00<00:00, 575.75it/s]


...

100%|██████████| 16/16 [00:12<00:00,  1.27it/s]


=== Document parser messages ===
Using RapidOCR for OCR processing.
OCR on page.number=2/3.


Generating markdown text...


100%|██████████| 16/16 [00:00<00:00, 727.20it/s]


Done: ngoho2026.pdf
  Figures  : 8
  Formulas : 1
  Markdown : D:\machine learning\datasets\WattBot2026\processed_markdown\ngoho2026\document.md

[4/4] vanderbauwhede2024.pdf

Processing: vanderbauwhede2024.pdf

=== Document parser messages ===
Using RapidOCR for OCR processing.
OCR on page.number=2/3.
Done: vanderbauwhede2024.pdf
  Figures  : 1
  Formulas : 0
  Markdown : D:\machine learning\datasets\WattBot2026\processed_markdown\vanderbauwhede2024\document.md

BATCH COMPLETE
Total     : 4
Successful: 4
Failed    : 0
